# Quantization intro

In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM
from getpass import getpass
from huggingface_hub import login

In [5]:
login(token=getpass("Hugging Face token: "), add_to_git_credential=False)


In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

Architecture: arm64
macOS: 15.2
PyTorch: 2.14.1
Device: mps


In [45]:
MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE,).to(device)

print(f"Model loaded on: {model.device}\n")
print(model)

Loading weights: 100%|██████████| 236/236 [00:01<00:00, 131.41it/s]


Model loaded on: mps:0

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RM

In [44]:
print(model)

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [15]:
def tokenize_recieve(prompt):
    message = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0, input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer

In [16]:
res = tokenize_recieve("Explain machine learning in 2 simple sentences.")
print(res)

Machine learning is a type of computer science that uses algorithms to learn from data without being explicitly programmed.



## Linear Quantization (symmetric and affine)

In [28]:
def symmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Quantize using one scale for the entire tensor.
    Zero point is fixed at 0.

    Returns:
        q: Integer tensor.
        scale: Scalar tensor.
    """
    max_original = x.abs().max()
    max_int = torch.iinfo(dtype).max
    s = max_original / max_int
    s = torch.where(s == 0, torch.ones_like(s), s)
    q = torch.round(x / s).clamp(-max_int, max_int).to(dtype)

    return q, s

def asymmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Quantize using one scale and zero point for the entire tensor.

    Returns:
        q: Integer tensor.
        scale: Scalar tensor.
        zero_point: Scalar integer tensor.
    """
    min_original, max_original = x.min().clamp(max=0), x.max().clamp(min=0)
    min_int, max_int = torch.iinfo(dtype).min, torch.iinfo(dtype).max

    s = (max_original - min_original) / (max_int - min_int)
    s = torch.where(s == 0, torch.ones_like(s), s)

    z = torch.round(min_int - min_original / s).clamp(min_int, max_int).to(dtype)
    q = (torch.round(x / s) + z.float()).clamp(min_int, max_int).to(dtype)

    return q, s, z

def dequantize(q: torch.Tensor, s: torch.Tensor, z: int | torch.Tensor = 0) -> torch.Tensor:
    """
    Reconstruct approximate floating-point values:
        x_hat = scale * (q - zero_point)
    """
    return s.float() * (q.float() - torch.as_tensor(z, dtype=torch.float32, device=q.device))